In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

In [ ]:
#import training data
file = pd.read_csv("development_unscaled.csv")

In [ ]:
file

In [ ]:
# Random Forest Classifier — Breast Cancer (Unscaled Data)

#This notebook trains a **Random Forest** model on `development_unscaled.csv` using **5-fold stratified cross-validation** and evaluates on the held-out `test_unscaled_FINAL_HOLDOUT.csv`.


In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score,
)
from imblearn.over_sampling import SMOTE

#That block just sets up file paths so the notebook knows where to read/write things.
DEV_PATH = "../../data/processed/development_unscaled.csv"
TEST_PATH = "../../data/processed/test_unscaled_FINAL_HOLDOUT.csv"

def load_xy(path):
    df = pd.read_csv(path)
    y = df["diagnosis"].astype(int)
    X = df.drop(columns=["id", "diagnosis"])
    return X, y

#assigning features (x) and labels (y) to our development and test path 
X_dev, y_dev = load_xy(DEV_PATH)
X_test, y_test = load_xy(TEST_PATH)
print("Development:", X_dev.shape, "| Holdout:", X_test.shape)
print(y_dev)

Development: (455, 30) | Holdout: (114, 30)
0      1
1      0
2      0
3      1
4      1
      ..
450    0
451    0
452    0
453    0
454    1
Name: diagnosis, Length: 455, dtype: int64


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"],
}

#Choosing the best model  
rf = RandomForestClassifier(random_state=42, class_weight="balanced_subsample")
grid = GridSearchCV(rf, param_grid, scoring="accuracy", cv=cv, n_jobs=1)
grid.fit(X_dev, y_dev)

print("Best params:", grid.best_params_)
best_model = grid.best_estimator_


#Best model with threshold tuning, and using SMOTE
pipe = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("rf", RandomForestClassifier(
        random_state=42,
        class_weight="balanced_subsample",
    )),
])


param_grid_2 = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [None, 5, 10, 20],
    "rf__min_samples_split": [2, 5],
    "rf__min_samples_leaf": [1, 2],
    "rf__max_features": ["sqrt", "log2"],
}

grid_2 = GridSearchCV(pipe, param_grid_2, scoring="recall", cv=cv, n_jobs=1)
grid_2.fit(X_dev, y_dev)
print("Best params2:", grid_2.best_params_)
best_model_2 = grid.best_estimator_


In [ ]:
cv_scores = cross_validate(
    best_model,
    X_dev,
    y_dev,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
    n_jobs=1,
)

print("5-fold CV accuracy:", cv_scores["test_accuracy"].mean(), "+/-", cv_scores["test_accuracy"].std())

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("\nHoldout accuracy:", accuracy_score(y_test, y_pred))
print("Holdout F1:", f1_score(y_test, y_pred))
print("Holdout MCC:", matthews_corrcoef(y_test, y_pred))
print("Holdout ROC-AUC:", roc_auc_score(y_test, y_proba))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\n", classification_report(y_test, y_pred))

In [ ]:
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump(best_model, MODEL_PATH)
print("Saved model:", MODEL_PATH)

feature_importance = pd.DataFrame({
    "feature": X_dev.columns,
    "importance": best_model.feature_importances_,
}).sort_values("importance", ascending=False)

feature_importance.head(10)

In [ ]:
import joblib

model = joblib.load(
    "/Users/chloegates/Machine Learning/Project Random Forrest/models/random_forest_cv5_model.joblib"
)

print(type(model))
print(model)

In [ ]:
#inspecting weights
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

# development labels
y = pd.read_csv("development_unscaled.csv")["diagnosis"].astype(int)

classes = np.unique(y)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

print("Classes:", classes)          # [0, 1] = benign, malignant
print("Weights:", weights)
print(dict(zip(classes, weights)))

In [ ]:
#Questions being asked in my decision trees
from sklearn.tree import export_text
import joblib

model = joblib.load("models/random_forest_cv5_model.joblib")

# first tree in the forest
print(export_text(
    model.estimators_[0],
    feature_names=list(model.feature_names_in_)
)[:2000])